[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bigbathtub101/sector-rotation-system/blob/main/notebooks/setup_and_backtest.ipynb)

# Sector Rotation System — Setup & Backtest

This notebook runs on **Google Colab** (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Runs the full pipeline (regime → optimizer → screener → NLP)
4. Executes a walk-forward backtest and displays results
5. Shows the current Executive Summary with dollar allocations

**Runtime:** ~5–8 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [13]:
# --- Clone private repo using Colab Secrets ---
# Store your GitHub PAT in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN.
# The token needs 'repo' scope for private repository access.

import os
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('\nSetup complete.')

Repo already cloned at /content/sector-rotation-system, pulling latest...
Already up to date.
Working directory: /content/sector-rotation-system

Setup complete.


## 2. Set API Keys (Optional)

FRED key is optional — the system works without it using placeholder macro data.
Set it here if you have one.

In [14]:
import os

# Optional: set your FRED API key (free at https://fred.stlouisfed.org)
os.environ['FRED_API_KEY'] = 'a5c36208f499faee4aff0b1d96019406'

# Verify
fred_key = os.environ.get('FRED_API_KEY', '')
print(f"FRED_API_KEY: {'set (' + fred_key[:4] + '...)' if fred_key else 'NOT SET (will use dummy macro data)'}")

FRED_API_KEY: set (a5c3...)


## 3. Load Config & Backfill Historical Data

Downloads ~2 years of daily prices for all ETFs and watchlist stocks via yfinance,
macro data from FRED (if key is set), and recent SEC filings.

In [15]:
import yaml
import sqlite3
import pandas as pd
from pathlib import Path

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ----- CRITICAL: Define a single DB path used by EVERYTHING -----
# All .py modules define  DB_PATH = Path(__file__).parent / 'rotation_system.db'
# at module scope.  In Colab, __file__ can resolve unpredictably, so we
# explicitly override every module's DB_PATH *and* patch default function
# arguments that captured the old value at import time.
DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

print("Config loaded. Portfolio total: ${:,.0f}".format(config['portfolio']['total_value']))
print(f"  Taxable: ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA: ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs: {len(config['tickers']['sector_etfs'])}")
print(f"  Watchlist tickers: {sum(len(config['tickers'].get(k, [])) for k in ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense', 'watchlist_green_materials'])}")
print(f"  DB path: {DB_PATH}")

Config loaded. Portfolio total: $144,284
  Taxable: $98,237
  Roth IRA: $46,047
  Sector ETFs: 11
  Watchlist tickers: 47
  DB path: /content/sector-rotation-system/rotation_system.db


In [16]:
# --- Helper: patch a module's DB_PATH everywhere it matters ---
import inspect

def patch_db_path(mod, new_path):
    """
    Overwrite a module's DB_PATH global, then walk its public functions
    and replace any default argument that was the OLD DB_PATH value
    (captured at import time) with the new path.
    """
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path

    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        sig = inspect.signature(obj)
        new_defaults = []
        changed = False
        if obj.__defaults__:
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)

    print(f"  {mod.__name__}.DB_PATH = {mod.DB_PATH}")

print("patch_db_path helper defined.")

patch_db_path helper defined.


In [17]:
# Run the full data ingestion pipeline
import data_feeds

# Patch DB_PATH AND the init_database default argument
patch_db_path(data_feeds, DB_PATH)

print("\n=== Running full data ingestion (this takes 1-3 min) ===")
ingestion_result = data_feeds.run_full_ingestion(config)

print(f"\n=== Ingestion Summary ===")
print(f"  Prices: {ingestion_result.get('prices', {}).get('rows', 'N/A')} rows, "
      f"{ingestion_result.get('prices', {}).get('tickers', 'N/A')} tickers")
print(f"  Macro:  {ingestion_result.get('macro', {}).get('rows', 'N/A')} rows, "
      f"{ingestion_result.get('macro', {}).get('series', 'N/A')} series")
print(f"  Filings: {ingestion_result.get('filings', {}).get('count', 'N/A')}")
if ingestion_result.get('warnings'):
    print(f"  Warnings: {len(ingestion_result['warnings'])}")
    for w in ingestion_result['warnings'][:5]:
        print(f"    - {w}")

# Verify the database was actually created with data
print(f"\n=== DB Verification ===")
print(f"  DB file exists: {DB_PATH.exists()}")
if DB_PATH.exists():
    print(f"  DB file size: {DB_PATH.stat().st_size:,} bytes")
    conn = sqlite3.connect(str(DB_PATH))
    for table in ['prices', 'macro_data', 'filings', 'signals']:
        try:
            count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn).iloc[0]['n']
            print(f"  {table}: {count:,} rows")
        except Exception as e:
            print(f"  {table}: ERROR - {e}")
    conn.close()
else:
    print("  ERROR: DB file was NOT created! Check ingestion logs above.")

  data_feeds.DB_PATH = /content/sector-rotation-system/rotation_system.db

=== Running full data ingestion (this takes 1-3 min) ===


[*********************100%***********************]  79 of 79 completed



=== Ingestion Summary ===
  Prices: 39579 rows, 79 tickers
  Macro:  616 rows, 6 series
  Filings: N/A
  Warnings: 1
    - STALE_PRICES: 1 tickers backfilled on 2026-02-27: CYBR

=== DB Verification ===
  DB file exists: True
  DB file size: 28,790,784 bytes
  prices: 39,579 rows
  macro_data: 616 rows
  filings: 440 rows
  signals: 496 rows


## 4. Detect Current Regime

In [18]:
import regime_detector

# Patch DB_PATH so regime_detector reads from the same DB
patch_db_path(regime_detector, DB_PATH)

regime_result = regime_detector.run_regime_detection(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('dominant_regime', 'N/A')}")
print(f"  Wedge Volume:        {regime_result.get('wedge_volume', 'N/A')}")
print(f"  Percentile:          {regime_result.get('percentile', 'N/A')}")
print(f"  Fast Shock Active:   {regime_result.get('fast_shock', False)}")
print(f"  Confidence:          {regime_result.get('confidence', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days', 'N/A')}")

  regime_detector.DB_PATH = /content/sector-rotation-system/rotation_system.db
CURRENT REGIME
  Regime:              offense
  Wedge Volume:        N/A
  Percentile:          N/A
  Fast Shock Active:   False
  Confidence:          N/A
  Consecutive Days:    N/A


## 5. Run CVaR Optimization

In [19]:
import portfolio_optimizer

# Patch DB_PATH
patch_db_path(portfolio_optimizer, DB_PATH)

regime = regime_result.get('dominant_regime', 'offense')
opt_result = portfolio_optimizer.run_portfolio_optimization(cfg=config, regime=regime)

total = config['portfolio']['total_value']
taxable = config['portfolio']['accounts']['taxable']['value']
roth = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 60)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime})")
print("=" * 60)

positions = opt_result.get('positions', {})
if positions:
    print(f"{'Ticker':<10} {'Weight':>8} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}")
    print("-" * 50)
    for ticker, info in sorted(positions.items(), key=lambda x: -x[1].get('pct', 0)):
        pct = info.get('pct', 0) / 100.0
        total_d = pct * total
        tax_d = pct * taxable
        roth_d = pct * roth
        print(f"{ticker:<10} {pct:>7.1%} {total_d:>10,.0f} {tax_d:>10,.0f} {roth_d:>10,.0f}")
else:
    print("No positions returned. Check optimizer logs above.")

  portfolio_optimizer.DB_PATH = /content/sector-rotation-system/rotation_system.db


/content/sector-rotation-system/portfolio_optimizer.py:68: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader(


CVaR-OPTIMIZED ALLOCATION (Regime: offense)
Ticker       Weight    Total $  Taxable $     Roth $
--------------------------------------------------
BIL          15.0%     21,643     14,736      6,907
VGK          15.0%     21,643     14,736      6,907
XLV          10.0%     14,428      9,824      4,605
XLC           6.0%      8,643      5,884      2,758
XLF           6.0%      8,643      5,884      2,758
XLP           6.0%      8,643      5,884      2,758
XLRE          6.0%      8,643      5,884      2,758
XLU           6.0%      8,643      5,884      2,758
XLY           6.0%      8,643      5,884      2,758
XLB           5.0%      7,214      4,912      2,302
XLE           5.0%      7,214      4,912      2,302
XLI           3.0%      4,329      2,947      1,381
XLK           3.0%      4,329      2,947      1,381
INDA          1.4%      2,006      1,365        640
EEM           1.3%      1,919      1,307        612
EWT           1.3%      1,919      1,307        612
EWY           1.3%  

/usr/local/lib/python3.12/dist-packages/pypfopt/expected_returns.py:32: UserWarning: Some returns are NaN. Please check your price data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pypfopt/expected_returns.py:36: UserWarning: Some returns are infinite. Please check your price data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:56: RuntimeWarning: invalid value encountered in reduce
  return umr_prod(a, axis, dtype, out, keepdims, initial, where)


## 6. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly
based on the regime detected at each rebalance date.

In [20]:
import numpy as np
import json as json_module

# Use the same DB_PATH for direct SQL queries too
conn = sqlite3.connect(str(DB_PATH))

# Load SPY as benchmark
spy = pd.read_sql(
    "SELECT date, close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['close'].pct_change().dropna()

# Load signals for regime history
signals = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime' ORDER BY date",
    conn, parse_dates=['date']
)
conn.close()

if len(signals) > 0 and len(spy_returns) > 0:
    signals['regime'] = signals['signal_data'].apply(
        lambda x: json_module.loads(x).get('dominant_regime', 'offense') if isinstance(x, str) else 'offense'
    )

    regime_exposure = {'offense': 0.75, 'defense': 0.35, 'panic': 0.05}

    signals = signals.set_index('date')
    merged = spy_returns.to_frame('spy_ret').join(signals['regime'], how='left')
    merged['regime'] = merged['regime'].ffill().fillna('offense')
    merged['exposure'] = merged['regime'].map(regime_exposure)
    merged['strategy_ret'] = merged['spy_ret'] * merged['exposure']

    merged['spy_cum'] = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()

    days = len(merged)
    ann_factor = 252 / days if days > 0 else 1

    spy_total = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1

    spy_ann = (1 + spy_total) ** ann_factor - 1
    strat_ann = (1 + strat_total) ** ann_factor - 1

    spy_vol = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol = merged['strategy_ret'].std() * np.sqrt(252)

    spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0

    def max_dd(cum_series):
        peak = cum_series.cummax()
        dd = (cum_series - peak) / peak
        return dd.min()

    spy_mdd = max_dd(merged['spy_cum'])
    strat_mdd = max_dd(merged['strategy_cum'])

    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw = strat_ann - spy_ann
    alpha_adj = alpha_raw * mclean_decay
    mp_label = config['backtest']['mclean_pontiff_label']

    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    print(f"Period: {merged.index[0].strftime('%Y-%m-%d')} to {merged.index[-1].strftime('%Y-%m-%d')} ({days} trading days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {spy_mdd:>14.1%} {strat_mdd:>14.1%}")
    print()
    print(f"Raw Alpha vs SPY:      {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (x{mclean_decay}): {alpha_adj:>+.2%}")
    print(f"  {mp_label}")
else:
    print("Not enough data for backtest. Run data ingestion first.")

Not enough data for backtest. Run data ingestion first.


## 7. Stock Screener — Top Picks

In [21]:
import stock_screener

# Patch DB_PATH
patch_db_path(stock_screener, DB_PATH)

screener_result = stock_screener.run_stock_screener(cfg=config, regime=regime)

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

watchlists = screener_result.get('watchlists', {})
for name, df in watchlists.items():
    if df is not None and not (hasattr(df, 'empty') and df.empty):
        print(f"\n--- {name.upper()} ---")
        if hasattr(df, 'to_string'):
            print(df.head(5).to_string())
        else:
            print(df)

  stock_screener.DB_PATH = /content/sector-rotation-system/rotation_system.db
THEMATIC WATCHLIST SCORES


## 8. Run Full Monitor (Executive Summary)

In [22]:
# Run the full monitor pipeline and display the executive summary.
# --no-deliver skips Telegram/email/Sheets delivery (not configured in Colab).
# --db explicitly passes our DB path to avoid any path mismatch.
!python monitor.py --no-deliver --db "{DB_PATH}" 2>&1 | head -120

2026-02-28 12:32:42,745 [INFO] monitor: Configuration loaded from /content/sector-rotation-system/config.yaml
2026-02-28 12:32:42,746 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-02-28 12:32:42,746 [INFO] monitor:   MONITOR RUN: run_20260228_123242
2026-02-28 12:32:42,746 [INFO] monitor:   Date: 2026-02-28   Mode: LIVE
2026-02-28 12:32:42,746 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-02-28 12:32:42,746 [INFO] monitor: Step 1: Data refresh...
2026-02-28 12:32:42,941 [WARNING] monitor: data_feeds not available — skipping data refresh.
2026-02-28 12:32:42,941 [INFO] monitor:   Data refresh: skip
2026-02-28 12:32:42,941 [INFO] monitor: Step 2-3: Regime detection...
2026-02-28 12:32:42,942 [WARNING] monitor: regime_detector not available — using last known regime.
2026-02-28 12:32:42,942 [INFO] monitor:   Regime: offense (confirmed: True)
2026-02-28 12:32:42,942 [INFO] monitor: Step 4: Optimizer check...
2026-02-28

## 9. Download Database

Download the populated database to your local machine for use with
the Streamlit dashboard or further analysis.

In [23]:
from google.colab import files

# Download the populated database
files.download(str(DB_PATH))
print("Database downloaded. Place it in the repo root for the dashboard.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Database downloaded. Place it in the repo root for the dashboard.
